<h1><center>Lab: Cloud APIS and AI News Briefing Generator</center></h1>

## Load and Test API Keys

In [5]:
import os
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Helper function to render LLM output as formatted markdown
def md(text):
    """ Display text as rendered markdown."""
    display(Markdown(text))

# Load environment variables from .env file
load_dotenv()

# Quick check that keys are loaded
openai_key = os.getenv('OPENAI_API_KEY')
anthropic_key = os.getenv('ANTHROPIC_API_KEY')

def mask_key(key):
    """ Safely mask an API key for display """
    if not key:
        return 'NOT SET'
    if len(key) < 12:
        return '****'
    return key[:8] + '...' + key[-4:]

print('API Key Status')
print(f"    OpenAI:      {mask_key(openai_key)}")
print(f"    Anthropic:   {mask_key(anthropic_key)}")
print("\n Setup Complete")


API Key Status
    OpenAI:      sk-proj-...SSAA
    Anthropic:   sk-ant-u...rQAA

 Setup Complete


## Initialize and Test OpenAI Client

In [6]:
from openai import OpenAI

#Initialize client - automatically reads OPENAI_API_KEY
client = OpenAI()
print("OpenAI client initialized successfully.")

# Test the connection
def test_openai():
    """ Test the OpenAI connection """
    print("\n Testing OpenAI Connection...")
    print("_" * 50)

    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {
                    'role': 'user',
                    'content': "Say 'Hello, AI Engineer!' in exactly those words."
                }
            ],
            max_tokens = 20,
            temperature=0
        )
        print("Connection Successful")
        print(f"Response: {response.choices[0].message.content}")
        print(f"Tokens Used: {response.usage.total_tokens}")
        return True

    except Exception as e:
        print(f" Connection Failed: {type(e).__name__}")
        print(f" Error: {str(e)[:200]}")
        return False
test_openai()

OpenAI client initialized successfully.

 Testing OpenAI Connection...
__________________________________________________
Connection Successful
Response: Hello, AI Engineer!
Tokens Used: 24


True

## Initialize and Test Claude Client

In [7]:
import anthropic

# Initialize Claude client
anthropic_client = anthropic.Anthropic()
print("Claude client initialized successfully")

# Test the connection
def test_claude():
    """ Test the Claude connection """
    print("\n Testing Claude Connection...")
    print("_" * 50)

    try:
        # Try latest model first, then fall back
        claude_models = ["claude-sonnet-5", "claude-sonnet-4-6"]
        
        for model_name in claude_models:
            try:
                response = anthropic_client.messages.create(
                    model=model_name,
                    max_tokens=30,
                    messages=[
                        {
                            'role': 'user',
                            'content': 'Say "Hello, AI Engineer" in these words exactly.'
                        }
                        ]
                )
                print(f"Connection Successful (model): {model_name}")
                print(f"Response: {response.content[0].text}")
                print(f"Tokens used: {response.usage.input_tokens + response.usage.output_tokens}")
                return True, model_name
            except Exception:
                continue
        print("No compatible Claude model found.")
        return False, None
    
    except Exception as e:
        print(f"Connection Failed: {type(e).__name__}")
        print(f"Error message: {str(e)[:200]}")
        return False, None
claude_ok, claude_model = test_claude()

Claude client initialized successfully

 Testing Claude Connection...
__________________________________________________
Connection Successful (model): claude-sonnet-5
Response: Hello, AI Engineer
Tokens used: 39


<h1><center>Project: News Briefing Generator</center></h1>

## Build Morning Briefing Dashboard

In [8]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_website(url, selector='a', num_stories=10):
    """ Scrape headlines from any news website.
    
    Args:
        url: The webpage URL to scrape 
        selector: CSS selector to find the headline links (default: "a")
        num_stories: How many stories to return (default: 10)

    Returns:
        List of dicts with rank, title, link, and source
    """
    print(f"Fetching {url}...")
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    stories = []
    for idx, item in enumerate(soup.select(selector)[:num_stories], start=1):
        title = item.text.strip()
        link = item.get("herf","")
        # Handle relative URLs
        if link and not link.startswith("http"):
            link = urljoin(url, link)
        if title:
            stories.append({
                'rank': str(idx),
                'title': title,
                'link': link,
                'source': url
            })
    return stories

# --- Scrape NPR (main example) ---
# To try a different site, swap the url and selector below.
# Examples:
# scrape_website("https://lite.cnn.com/", selector="li > a")
# scrape_website("https://lobste.rs/", selector="a.u-url")
# scrape_website("https://68k.news", selector="li > a")

stories = scrape_website(
    url="https://text.npr.org/",
    selector="a.topic-title",
    num_stories=10
)

print(f"\n{'_'*50}")
print(f"Scraped {len(stories)} stories from {stories[0]['source'] if stories else'N/A'}")
for s in stories:
    print(f" {s['rank']:>2}. {s['title']}")


Fetching https://text.npr.org/...

__________________________________________________
Scraped 10 stories from https://text.npr.org/
  1. Democrats — and Elon Musk — continue to dominate midterm campaign fundraising
  2. Beef prices are on the rise. Here's how it is affecting Texans this election season
  3. Why these 5 everyday items fluctuate in price at the grocery store
  4. TikTok staff raised child safety alarms. Court records show executives failed to act
  5. Can the new federal voucher program help public schools? Democrats are torn
  6. Vanessa Kerry is the new WHO envoy for health resilience. What's on her to-do list?
  7. Between the coasts, populist Democrats see room to reclaim lost ground in red states
  8. Elizabeth Holmes and Nathan Fielder are both master manipulators in this documentary
  9. The new Elizabeth Holmes film blends fact, fiction and fraud
 10. Rays will face Brewers in World Series to produce a first-time champion


## Generate Morning Brief with OpenAI

In [9]:
# Prepare headlines as plain text for the LLM
headlines_text = "\n".join([f"{s['rank']}. {s['title']}" for s in stories])

print("Sending headlines to OpenAI...\n")

# Generate briefing with OpenAI
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            'role': 'system',
            'content':(
                "You are a news editor writing a concise morning briefing."
                "Be informative and highlight what matters most."
            )
        },
        {
            'role': 'user',
            'content':(
                f"Here are today's top stories: \n\n"
                f"{headlines_text}\n\n"
                f"Write a 2-3 paragraph morning briefing summarizing the key themes "
                f" and most important stories. Use markdown formatting."
            )
        }
    ],
    max_tokens=500,
    temperature=0.7
)

openai_briefing = response.choices[0].message.content
md(f"## OpenAI's Morning Briefing\n\n{openai_briefing}")

Sending headlines to OpenAI...



## OpenAI's Morning Briefing

### Morning Briefing: Key Highlights for Today

In the lead-up to the midterm elections, campaign fundraising efforts are showing a significant advantage for Democrats, bolstered by notable contributions from figures like Elon Musk. As candidates prepare for the polls, rising beef prices are becoming a critical issue, particularly in Texas, influencing voter sentiment and economic discussions in the region. Additionally, the fluctuation of grocery prices on everyday items is drawing attention, prompting concerns about inflation and its impact on households across the country.

In other news, TikTok is facing scrutiny after internal staff raised alarms regarding child safety, with court records indicating a lack of executive action on the matter. Meanwhile, the newly implemented federal voucher program is sparking debate among Democrats regarding its potential to aid public schools. On the global front, Vanessa Kerry has been appointed as the WHO envoy for health resilience, taking on a significant agenda for public health improvement. Lastly, in sports, the Tampa Bay Rays are set to face the Milwaukee Brewers in the World Series, marking a historic opportunity for both teams to secure their first championship title.

## Categorize Headlines With Claude

In [14]:
print("Sending headlines to Claude...\n")

# Use the model that worked during our test, or fall back to a default.
model_to_use = claude_model if claude_model else "claude-sonnet-4-6"

response = anthropic_client.messages.create(
    model=model_to_use,
    max_tokens=800,
    messages=[
        {
            'role': 'user',
            'content':(
                f"Here are today's top stories: \n\n"
                f"{headlines_text}\n\n"
                f"For each story:\n"
                f"1. Assign a category: AI, Web Dev, Security, Business, Science, Politics, Culture, or Other\n"
                f"2. Rate its relevance for an AI engineer: High, Medium, or Low\n\n"
                f"Present this as a markdown table with columns: Title, Category, Relevance.\n\n"
                f"Then pick the number 1 must-read story for an AI engineer and explain why in 2 sentences."

            )
        }
    ]
)

text_blocks = [block.text for block in response.content if block.type == "text"]
if not text_blocks:
    raise ValueError(
        f"Claude returned no text blocks (stop_reason={response.stop_reason!r}). "
        "Inspect response.content for available block types."
    )
claude_analysis = "\n".join(text_blocks)
md(f"## Claude's Analysis\n\n{claude_analysis}")

Sending headlines to Claude...



## Claude's Analysis

| Title | Category | Relevance |
|---|---|---|
| Democrats — and Elon Musk — continue to dominate midterm campaign fundraising | Politics | Low |
| Beef prices are on the rise. Here's how it is affecting Texans this election season | Business | Low |
| Why these 5 everyday items fluctuate in price at the grocery store | Business | Low |
| TikTok staff raised child safety alarms. Court records show executives failed to act | Security | Medium |
| Can the new federal voucher program help public schools? Democrats are torn | Politics | Low |
| Vanessa Kerry is the new WHO envoy for health resilience. What's on her to-do list? | Other | Low |
| Between the coasts, populist Democrats see room to reclaim lost ground in red states | Politics | Low |
| Elizabeth Holmes and Nathan Fielder are both master manipulators in this documentary | Culture | Low |
| The new Elizabeth Holmes film blends fact, fiction and fraud | Culture | Low |
| Rays will face Brewers in World Series to produce a first-time champion | Other | Low |

**Must-read for an AI engineer:** "TikTok staff raised child safety alarms. Court records show executives failed to act." This story touches on algorithmic recommendation systems, content moderation at scale, and the real-world consequences of AI-driven platform design — all directly relevant to engineers building or auditing ML systems with safety implications.

## AI Morning Briefing Dashboard

In [15]:
from datetime import datetime

today = datetime.now().strftime('%A, %B %d, %Y')
source_name = stories[0]['source'] if stories else 'News'

# Build headline links
headline_links = "\n\n".join([
    f"**{s['rank']}.** [{s['title']}]({s['link']})"
    for s in stories
])

# Display the complete briefing
md(f"""# Your Morning Briefing 
Powered by Live News + OpenAI + Claude

---

## Top Stories from {source_name}

{headline_links}

---

# Summary (GPT-4o-mini)

{openai_briefing}

---

## Story Analysis(Claude Sonnet)

{claude_analysis}

""")

# Your Morning Briefing 
Powered by Live News + OpenAI + Claude

---

## Top Stories from https://text.npr.org/

**1.** [Democrats — and Elon Musk — continue to dominate midterm campaign fundraising]()

**2.** [Beef prices are on the rise. Here's how it is affecting Texans this election season]()

**3.** [Why these 5 everyday items fluctuate in price at the grocery store]()

**4.** [TikTok staff raised child safety alarms. Court records show executives failed to act]()

**5.** [Can the new federal voucher program help public schools? Democrats are torn]()

**6.** [Vanessa Kerry is the new WHO envoy for health resilience. What's on her to-do list?]()

**7.** [Between the coasts, populist Democrats see room to reclaim lost ground in red states]()

**8.** [Elizabeth Holmes and Nathan Fielder are both master manipulators in this documentary]()

**9.** [The new Elizabeth Holmes film blends fact, fiction and fraud]()

**10.** [Rays will face Brewers in World Series to produce a first-time champion]()

---

# Summary (GPT-4o-mini)

### Morning Briefing: Key Highlights for Today

In the lead-up to the midterm elections, campaign fundraising efforts are showing a significant advantage for Democrats, bolstered by notable contributions from figures like Elon Musk. As candidates prepare for the polls, rising beef prices are becoming a critical issue, particularly in Texas, influencing voter sentiment and economic discussions in the region. Additionally, the fluctuation of grocery prices on everyday items is drawing attention, prompting concerns about inflation and its impact on households across the country.

In other news, TikTok is facing scrutiny after internal staff raised alarms regarding child safety, with court records indicating a lack of executive action on the matter. Meanwhile, the newly implemented federal voucher program is sparking debate among Democrats regarding its potential to aid public schools. On the global front, Vanessa Kerry has been appointed as the WHO envoy for health resilience, taking on a significant agenda for public health improvement. Lastly, in sports, the Tampa Bay Rays are set to face the Milwaukee Brewers in the World Series, marking a historic opportunity for both teams to secure their first championship title.

---

## Story Analysis(Claude Sonnet)

| Title | Category | Relevance |
|---|---|---|
| Democrats — and Elon Musk — continue to dominate midterm campaign fundraising | Politics | Low |
| Beef prices are on the rise. Here's how it is affecting Texans this election season | Business | Low |
| Why these 5 everyday items fluctuate in price at the grocery store | Business | Low |
| TikTok staff raised child safety alarms. Court records show executives failed to act | Security | Medium |
| Can the new federal voucher program help public schools? Democrats are torn | Politics | Low |
| Vanessa Kerry is the new WHO envoy for health resilience. What's on her to-do list? | Other | Low |
| Between the coasts, populist Democrats see room to reclaim lost ground in red states | Politics | Low |
| Elizabeth Holmes and Nathan Fielder are both master manipulators in this documentary | Culture | Low |
| The new Elizabeth Holmes film blends fact, fiction and fraud | Culture | Low |
| Rays will face Brewers in World Series to produce a first-time champion | Other | Low |

**Must-read for an AI engineer:** "TikTok staff raised child safety alarms. Court records show executives failed to act." This story touches on algorithmic recommendation systems, content moderation at scale, and the real-world consequences of AI-driven platform design — all directly relevant to engineers building or auditing ML systems with safety implications.

